# Edu Game AI Service - Graph Pipeline Testing

Notebook kiểm thử LangGraph pipeline cho dự án edu-game-ai-service.

## Mục tiêu
1. Kiểm tra imports và configuration
2. Test tạo GenerationRequest
3. Test từng node riêng lẻ
4. Test full graph pipeline
5. Kiểm tra kết quả output

---

## 1. Setup & Imports

In [1]:
# Add project root to path
import sys
import os
sys.path.insert(0, "..")

# ========================================
# LangSmith Configuration (Optional)
# ========================================
# Để theo dõi execution qua LangSmith, uncomment và nhập API key:
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "edu-game-ai-dev"
# os.environ["LANGCHAIN_API_KEY"] = "lsv2_pt_XXXXXXXXXXXXXXXX"  # Replace with your key from https://smith.langchain.com/settings
# os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"

# Import configuration
from src.config import get_settings

# Verify settings load
settings = get_settings()
print(f"✅ Settings loaded")
print(f"   Project: {settings.gcp_project_id}")
print(f"   Location: {settings.gcp_location}")
print(f"   Data Store: {settings.data_store_id}")
print(f"   Bucket: {settings.gcs_bucket}")

# Check if LangSmith tracing is enabled
if os.getenv("LANGCHAIN_TRACING_V2") == "true":
    print(f"✅ LangSmith tracing enabled")
    print(f"   Project: {os.getenv('LANGCHAIN_PROJECT', 'default')}")
else:
    print("ℹ️  LangSmith tracing disabled (uncomment code above to enable)")

✅ Settings loaded
   Project: green-mercury-485016-n1
   Location: asia-southeast1
   Data Store: aiservice-datastore-m1_1772802306291
   Bucket: documents-development-bucket
ℹ️  LangSmith tracing disabled (uncomment code above to enable)


In [2]:
# Import schemas
from src.api.schemas import (
    GameType, 
    DifficultyLevel, 
    GenerationRequest, 
    DocScope,
    GameContentResponse
)

# Import graph components
from src.graph.state import AgentState
from src.graph.builder import create_graph, compile_graph

print("✅ Schemas imported")
print("✅ Graph components imported")

✅ Schemas imported
✅ Graph components imported


## 2. Tạo Test Request

Tạo một `GenerationRequest` với topic "Đạo hàm" (Derivatives) để test pipeline.

In [3]:
# Tạo test request
test_request = GenerationRequest(
    user_id="test_user_001",
    topic="Đạo hàm",  # Derivatives
    game_types=[GameType.QUIZ, GameType.FLASHCARD],
    num_questions=3,  # Số nhỏ để test nhanh
    difficulty=DifficultyLevel.MEDIUM,
    doc_scope=DocScope.SYSTEM,  # Dùng system docs đã upload
    language="vi"
)

print("✅ Test request created:")
print(test_request.model_dump_json(indent=2))

✅ Test request created:
{
  "user_id": "test_user_001",
  "document_ids": null,
  "topic": "Đạo hàm",
  "game_types": [
    "quiz",
    "flashcard"
  ],
  "num_questions": 3,
  "difficulty": "medium",
  "doc_scope": "system",
  "language": "vi"
}


## 3. Create và Compile Graph

In [4]:
# Create and compile graph
graph = create_graph()
app = compile_graph()

print("✅ Graph created and compiled")
print(f"   Graph type: {type(graph).__name__}")
print(f"   App type: {type(app).__name__}")

2026-03-08 15:25:57 [info     ] creating_graph                
2026-03-08 15:25:57 [info     ] graph_created                 
2026-03-08 15:25:57 [info     ] creating_graph                
2026-03-08 15:25:57 [info     ] graph_created                 
2026-03-08 15:25:57 [info     ] graph_compiled_without_checkpointer
✅ Graph created and compiled
   Graph type: StateGraph
   App type: CompiledStateGraph


## 4. Run Full Pipeline

⚠️ **Lưu ý:** Cell này gọi Vertex AI và Gemini API → tốn quota/cost.

In [5]:
# Prepare initial state
initial_state = {
    "request": test_request,
    "doc_scope": test_request.doc_scope.value,
}

# Run the pipeline (async)
# Note: In Jupyter, we use await instead of asyncio.run() to avoid event loop conflicts
try:
    # Try to get existing event loop (Jupyter)
    import asyncio
    result = await app.ainvoke(initial_state)
    print("✅ Pipeline completed!")
except RuntimeError as e:
    if "no running event loop" in str(e):
        # Fallback for non-Jupyter environments
        result = asyncio.run(app.ainvoke(initial_state))
        print("✅ Pipeline completed!")
    else:
        raise

2026-03-08 15:26:05 [info     ] supervisor_analyzing           doc_scope=system topic='Đạo hàm' user_id=test_user_001


/home/sakana/Code/ManifoldsTeam/AIServices/notebooks/../src/graph/nodes/supervisor.py:61: LangChainDeprecationWarning: The class `ChatVertexAI` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An updated version of the class exists in the `langchain-google-genai package and should be used instead. To use it run `pip install -U `langchain-google-genai` and import as `from `langchain_google_genai import ChatGoogleGenerativeAI``.
  return ChatVertexAI(
Retrying langchain_google_vertexai.chat_models._acompletion_with_retry.<locals>._completion_with_retry_inner in 4 seconds as it raised NotFound: 404 Publisher Model `projects/green-mercury-485016-n1/locations/asia-southeast1/publishers/google/models/gemini-2.0-flash` was not found or your project does not have access to it. Please ensure you are using a valid model version. For more information, see: https://cloud.google.com/vertex-ai/generative-ai/docs/learn/model-versions.
Retrying langchain_google_vertexai.chat_models._aco

2026-03-08 15:26:37 [error    ] supervisor_classification_failed error='404 Publisher Model `projects/green-mercury-485016-n1/locations/asia-southeast1/publishers/google/models/gemini-2.0-flash` was not found or your project does not have access to it. Please ensure you are using a valid model version. For more information, see: https://cloud.google.com/vertex-ai/generative-ai/docs/learn/model-versions'
2026-03-08 15:26:37 [info     ] supervisor_classified          content_type=math
2026-03-08 15:26:37 [info     ] routing_to_agent               content_type=math route=math_agent
2026-03-08 15:26:37 [info     ] math_agent_starting            doc_scope=system num_questions=3 topic='Đạo hàm' user_id=test_user_001
2026-03-08 15:26:37 [info     ] retrieving_context             doc_scope=system query='Đạo hàm medium level educational content' user_id=test_user_001
2026-03-08 15:26:37 [debug    ] creating_retriever             data_store_id=aiservice-datastore-m1_1772802306291 doc_scope=syste

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-08 15:26:38 [error    ] context_retrieval_failed       error='400 Cannot use enterprise edition features (website search, multi-modal search, extractive answers/segments, etc.) in a standard edition search engine. Please follow https://cloud.google.com/generative-ai-app-builder/docs/enterprise-edition#toggle-enterprise to enable Enterprise edition. Enterprise edition can be only enabled at the engine / app level. If the search request is against a data store, please update the serving config in the search request to use the engine/app ID instead, in the format of: "projects/*/locations/*/collections/*/engines/*/servingConfigs/*".'
2026-03-08 15:26:38 [info     ] math_agent_context_retrieved   context_chunks=0


...

2026-03-08 15:28:42 [error    ] supervisor_classification_failed error='404 Publisher Model `projects/green-mercury-485016-n1/locations/asia-southeast1/publishers/google/models/gemini-2.0-flash` was not found or your project does not have access to it. Please ensure you are using a valid model version. For more information, see: https://cloud.google.com/vertex-ai/generative-ai/docs/learn/model-versions'
2026-03-08 15:28:42 [info     ] supervisor_classified          content_type=math
2026-03-08 15:28:42 [info     ] routing_to_agent               content_type=math route=math_agent
2026-03-08 15:28:42 [info     ] math_agent_starting            doc_scope=system num_questions=3 topic='Đạo hàm' user_id=test_user_001
2026-03-08 15:28:42 [info     ] retrieving_context             doc_scope=system query='Đạo hàm medium level educational content' user_id=test_user_001
2026-03-08 15:28:42 [debug    ] creating_retriever             data_store_id=aiservice-datastore-m1_1772802306291 doc_scope=syste

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-08 15:28:43 [error    ] context_retrieval_failed       error='400 Cannot use enterprise edition features (website search, multi-modal search, extractive answers/segments, etc.) in a standard edition search engine. Please follow https://cloud.google.com/generative-ai-app-builder/docs/enterprise-edition#toggle-enterprise to enable Enterprise edition. Enterprise edition can be only enabled at the engine / app level. If the search request is against a data store, please update the serving config in the search request to use the engine/app ID instead, in the format of: "projects/*/locations/*/collections/*/engines/*/servingConfigs/*".'
2026-03-08 15:28:43 [info     ] math_agent_context_retrieved   context_chunks=0


Retrying langchain_google_vertexai.chat_models._acompletion_with_retry.<locals>._completion_with_retry_inner in 4 seconds as it raised NotFound: 404 Publisher Model `projects/green-mercury-485016-n1/locations/asia-southeast1/publishers/google/models/gemini-2.0-flash` was not found or your project does not have access to it. Please ensure you are using a valid model version. For more information, see: https://cloud.google.com/vertex-ai/generative-ai/docs/learn/model-versions.
Retrying langchain_google_vertexai.chat_models._acompletion_with_retry.<locals>._completion_with_retry_inner in 4 seconds as it raised NotFound: 404 Publisher Model `projects/green-mercury-485016-n1/locations/asia-southeast1/publishers/google/models/gemini-2.0-flash` was not found or your project does not have access to it. Please ensure you are using a valid model version. For more information, see: https://cloud.google.com/vertex-ai/generative-ai/docs/learn/model-versions.
Retrying langchain_google_vertexai.chat_

2026-03-08 15:29:14 [error    ] math_agent_generation_failed   error='404 Publisher Model `projects/green-mercury-485016-n1/locations/asia-southeast1/publishers/google/models/gemini-2.0-flash` was not found or your project does not have access to it. Please ensure you are using a valid model version. For more information, see: https://cloud.google.com/vertex-ai/generative-ai/docs/learn/model-versions'
2026-03-08 15:29:14 [warning  ] reviewer_no_items_to_review   
2026-03-08 15:29:14 [info     ] review_router_deciding         iteration=3 max_iterations=3 pass_rate=0 rejected=0 reviewed=0
2026-03-08 15:29:14 [warning  ] review_router_pass_max_iterations required=2 reviewed=0
2026-03-08 15:29:14 [info     ] formatter_starting             game_types=['quiz', 'flashcard'] items_count=0 user_id=test_user_001
2026-03-08 15:29:14 [warning  ] formatter_no_items            
✅ Pipeline completed!


## 5. Kiểm tra kết quả

In [6]:
# Kiểm tra các trường trong kết quả
print("📊 State keys:", list(result.keys()))
print()

# Content items generated
content_items = result.get("content_items", [])
print(f"📝 Content items generated: {len(content_items)}")

# Reviewed items
reviewed_items = result.get("reviewed_items", [])
print(f"✅ Reviewed items passed: {len(reviewed_items)}")

# Rejected items
rejected_items = result.get("rejected_items", [])
print(f"❌ Rejected items: {len(rejected_items)}")

# Search context
search_context = result.get("search_context", [])
print(f"🔍 Search context chunks: {len(search_context)}")

📊 State keys: ['request', 'doc_scope', 'content_type', 'search_context', 'search_sources', 'content_items', 'reviewed_items', 'rejected_items', 'iteration_count', 'final_output', 'errors']

📝 Content items generated: 0
✅ Reviewed items passed: 0
❌ Rejected items: 0
🔍 Search context chunks: 0


In [7]:
# Kiểm tra final output
final_output = result.get("final_output")

if final_output:
    print("🎮 Final Output:")
    print(f"   Request ID: {final_output.request_id}")
    print(f"   User ID: {final_output.user_id}")
    print(f"   Generated at: {final_output.generated_at}")
    print()
    
    # Game content by type
    for game_type, items in final_output.content.items():
        print(f"   {game_type}: {len(items)} items")
    
    print()
    print("📈 Metadata:")
    print(f"   Total generated: {final_output.metadata.total_generated}")
    print(f"   Passed review: {final_output.metadata.total_passed_review}")
    print(f"   Rejected: {final_output.metadata.total_rejected}")
    print(f"   Generation time: {final_output.metadata.generation_time_seconds:.2f}s")
else:
    print("⚠️ No final output - check errors")
    print("Errors:", result.get("errors", []))

🎮 Final Output:
   Request ID: 8c4c67ec-be65-4728-bbb5-c33870b24e9b
   User ID: test_user_001
   Generated at: 2026-03-08 08:29:14.353158

   quiz: 0 items
   flashcard: 0 items

📈 Metadata:
   Total generated: 0
   Passed review: 0
   Rejected: 0
   Generation time: 0.00s


In [ ]:
# Hiển thị mẫu quiz question
import json

if final_output and "quiz" in final_output.content:
    quizzes = final_output.content["quiz"]
    if quizzes:
        print("📋 Sample Quiz Question:")
        print(json.dumps(quizzes[0], ensure_ascii=False, indent=2))
else:
    print("No quiz questions generated")